In [1]:
# SSL certificate verification issues on macOS
import ssl
import os

# MUST set these environment variables BEFORE importing transformers
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['SSL_CERT_FILE'] = ''

# Disable SSL verification in Python's ssl module
ssl._create_default_https_context = ssl._create_unverified_context

# Disable warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Patch huggingface_hub to disable SSL verification
import huggingface_hub
from huggingface_hub import constants
constants.HF_HUB_DISABLE_SSL_VERIFY = True

# Also patch the backend
from requests import Session
original_request = Session.request
def patched_request(self, *args, **kwargs):
    kwargs['verify'] = False
    return original_request(self, *args, **kwargs)
Session.request = patched_request

print("SSL verification disabled for HuggingFace downloads")

SSL verification disabled for HuggingFace downloads


In [2]:
from transformers import pipeline

# Create the pipeline
chatbot = pipeline("text2text-generation", model="facebook/blenderbot-400M-distill")

W0414 23:29:14.259000 60839 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Device set to use mps:0


In [3]:
conversation = chatbot("What have you been up to")

In [4]:
conversation

[{'generated_text': " I've been working on a project for work. It's a lot of work, but it's worth it in the end."}]

In [5]:
conversation = chatbot("I've been busy too - teaching an AI course")

In [6]:
conversation

[{'generated_text': " That's cool. What kind of course are you teaching? I'm a computer science major."}]

In [ ]:
conversation = chatbot("Do you like movies?")

In [ ]:
conversation

In [7]:
import gradio as gr

message_list = []
response_list = []

def favourite_chatbot(message, history):
    conversation = chatbot(message)
    
    return conversation[0]['generated_text']

demo_chatbot = gr.ChatInterface(favourite_chatbot, title="Favourite Chatbot", description="Enter text to start chatting.")

demo_chatbot.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
